In [2]:
import openpyxl 
from openpyxl.styles import Font, Alignment, PatternFill 
from openpyxl.utils import get_column_letter 
from openpyxl.chart import BarChart, Reference 
from openpyxl.chart.label import DataLabelList


# ---- CONFIG ---- 

SRC_FILE = "L:/2026 Pilar Plant Files/Requests from Departments/2024-2026 CC-MCC (CJ) v2.xlsx"   # must be in the same folder as this script 

OUT_FILE = "L:/2026 Pilar Plant Files/Requests from Departments/2024-2026 CC-MCC (CJ) v3 - with Dollars.xlsx"



wb = openpyxl.load_workbook(SRC_FILE)

 

bold = Font(bold=True) 

header_fill = PatternFill("solid", fgColor="D9E1F2") 

dollar_fmt = '$#,##0' 

pct_fmt = '0.0%' 

num_fmt = '#,##0'

 

# ================================================================= 

# CONFIRMED COLUMN MAP (from your actual file) 

# Detail Volume:  F=Group, G:AM = 33 months (Jan'24-Sep'26), AN = Avg Revenue 

# Detail Revenue: F=Group, G = Avg Revenue, H:AN = 33 months (Jan'24-Sep'26) 

# Summary by Month: B=Group, C:N=2024(12mo), O:Z=2025(12mo), AA:AI=2026 Jan-Sep 

# Post-EMR months = Y(Nov'25) Z(Dec'25) AA..AI (Jan-Sep '26) = 11 months 

# Summary: B=Pre-EMR%, C=Post-EMR Avg%, D:N = 11 monthly % cols (Nov'25-Sep'26) 

# =================================================================

 

# --------------------------------------------------------------- 

# STEP 1: Add helper "row total" columns (AO) to Detail Volume & Detail Revenue 

# --------------------------------------------------------------- 

dv = wb["Detail Volume"] 

dv["AO5"] = "Row Total Volume" 

dv["AO5"].font = bold 

for r in range(6, 673): 

    dv[f"AO{r}"] = f"=SUM(G{r}:AM{r})"

 

dr = wb["Detail Revenue"] 

dr["AO5"] = "Row Total $" 

dr["AO5"].font = bold 

for r in range(6, 673): 

    dr[f"AO{r}"] = f"=SUM(H{r}:AN{r})"

 

# --------------------------------------------------------------- 

# STEP 2: Rates tab — Full-Period Volume-Weighted Blended $/Case 

# --------------------------------------------------------------- 

if "Rates" in wb.sheetnames: 

    del wb["Rates"] 

rates = wb.create_sheet("Rates") 

rates["A1"] = "Blended $/Case Rates — Full Period (Jan 2024 - Sep 2026)" 

rates["A1"].font = Font(bold=True, size=12)

 

headers = ["Tier", "Total Volume", "Total Revenue ($)", "Blended Rate ($/Case)"] 

for i, h in enumerate(headers): 

    c = rates.cell(row=3, column=i+1, value=h) 

    c.font = bold 

    c.fill = header_fill

 

tiers = ["MCC", "CC", "W/O"] 

for i, tier in enumerate(tiers): 

    r = 4 + i 

    rates.cell(row=r, column=1, value=tier) 

    rates.cell(row=r, column=2, 

        value=f"=SUMIF('Detail Volume'!$F$6:$F$672,\"{tier}\",'Detail Volume'!$AO$6:$AO$672)") 

    rates.cell(row=r, column=3, 

        value=f"=SUMIF('Detail Revenue'!$F$6:$F$672,\"{tier}\",'Detail Revenue'!$AO$6:$AO$672)") 

    rates.cell(row=r, column=4, value=f"=C{r}/B{r}") 

    rates.cell(row=r, column=2).number_format = num_fmt 

    rates.cell(row=r, column=3).number_format = dollar_fmt 

    rates.cell(row=r, column=4).number_format = dollar_fmt

 

for col, w in zip("ABCD", [10, 16, 18, 20]): 

    rates.column_dimensions[col].width = w

 

# --------------------------------------------------------------- 

# STEP 3: Summary by Month - Dollars 

# --------------------------------------------------------------- 

if "Summary by Month - Dollars" in wb.sheetnames: 

    del wb["Summary by Month - Dollars"] 

smd = wb.create_sheet("Summary by Month - Dollars") 

smd["A1"] = "2024 - 2026 CC/MCC IP Visits — Financial Impact" 

smd["A1"].font = Font(bold=True, size=12) 

smd["A3"] = "Summary by Month - Dollars (Mix-Shift Lift vs. Pre-EMR Baseline)" 

smd["A3"].font = Font(bold=True)

 

months = ["Nov-25","Dec-25","Jan-26","Feb-26","Mar-26","Apr-26", 

          "May-26","Jun-26","Jul-26","Aug-26","Sep-26"] 

# Confirmed source columns in 'Summary by Month' tab for these 11 months 

src_cols = ["Y","Z","AA","AB","AC","AD","AE","AF","AG","AH","AI"] 

dest_cols = [get_column_letter(i+2) for i in range(11)]  # B..L 

total_col = "M"

 

smd["A5"] = "Grouping" 

smd["A5"].font = bold 

smd["A5"].fill = header_fill 

for dc, m in zip(dest_cols, months): 

    smd[f"{dc}5"] = m 

    smd[f"{dc}5"].font = bold 

    smd[f"{dc}5"].fill = header_fill 

smd[f"{total_col}5"] = "Total (11-Mo)" 

smd[f"{total_col}5"].font = bold 

smd[f"{total_col}5"].fill = header_fill

 

row_labels = { 

    6: "Actual Volume - MCC", 7: "Actual Volume - CC", 8: "Actual Volume - W/O", 

    9: "Total Volume (Actual)", 

    11: "Expected Volume - MCC (Pre-EMR Mix)", 12: "Expected Volume - CC (Pre-EMR Mix)", 

    13: "Expected Volume - W/O (Pre-EMR Mix)", 

    15: "Actual $ - MCC", 16: "Actual $ - CC", 17: "Actual $ - W/O", 

    18: "Total Actual $", 

    20: "Expected $ - MCC", 21: "Expected $ - CC", 22: "Expected $ - W/O", 

    23: "Total Expected $", 

    25: "Net Lift $ (Actual - Expected)", 

    26: "Cumulative Lift $", 

} 

for r, lbl in row_labels.items(): 

    smd[f"A{r}"] = lbl 

    if r in (9,18,23,25,26): 

        smd[f"A{r}"].font = bold

 

for i, (dc, sc) in enumerate(zip(dest_cols, src_cols)): 

    # Actual volumes (pull from Summary by Month tab: row7=MCC, row8=CC, row9=W/O) 

    smd[f"{dc}6"]  = f"='Summary by Month'!{sc}7" 

    smd[f"{dc}7"]  = f"='Summary by Month'!{sc}8" 

    smd[f"{dc}8"]  = f"='Summary by Month'!{sc}9" 

    smd[f"{dc}9"]  = f"=SUM({dc}6:{dc}8)"

 

    # Expected volumes at pre-EMR mix (Summary!B6=MCC%, B7=CC%, B8=W/O%) 

    smd[f"{dc}11"] = f"=Summary!$B$6*{dc}9" 

    smd[f"{dc}12"] = f"=Summary!$B$7*{dc}9" 

    smd[f"{dc}13"] = f"=Summary!$B$8*{dc}9"

 

    # Actual $ 

    smd[f"{dc}15"] = f"={dc}6*Rates!$D$4" 

    smd[f"{dc}16"] = f"={dc}7*Rates!$D$5" 

    smd[f"{dc}17"] = f"={dc}8*Rates!$D$6" 

    smd[f"{dc}18"] = f"=SUM({dc}15:{dc}17)"

 

    # Expected $ 

    smd[f"{dc}20"] = f"={dc}11*Rates!$D$4" 

    smd[f"{dc}21"] = f"={dc}12*Rates!$D$5" 

    smd[f"{dc}22"] = f"={dc}13*Rates!$D$6" 

    smd[f"{dc}23"] = f"=SUM({dc}20:{dc}22)"

 

    # Lift 

    smd[f"{dc}25"] = f"={dc}18-{dc}23" 

    if i == 0: 

        smd[f"{dc}26"] = f"={dc}25" 

    else: 

        prev = dest_cols[i-1] 

        smd[f"{dc}26"] = f"={prev}26+{dc}25"

 

    for r in (6,7,8,9,11,12,13): 

        smd[f"{dc}{r}"].number_format = num_fmt 

    for r in (15,16,17,18,20,21,22,23,25,26): 

        smd[f"{dc}{r}"].number_format = dollar_fmt

 

# Total column M 

vol_rows = [6,7,8,9,11,12,13] 

dollar_rows = [15,16,17,18,20,21,22,23,25] 

for r in vol_rows + dollar_rows: 

    smd[f"{total_col}{r}"] = f"=SUM(B{r}:L{r})" 

    smd[f"{total_col}{r}"].number_format = num_fmt if r in vol_rows else dollar_fmt 

smd[f"{total_col}26"] = "=L26" 

smd[f"{total_col}26"].number_format = dollar_fmt

 

smd.column_dimensions["A"].width = 34 

for dc in dest_cols + [total_col]: 

    smd.column_dimensions[dc].width = 13

 

# --------------------------------------------------------------- 

# STEP 3b: Net Lift $ chart 

# --------------------------------------------------------------- 

chart = BarChart() 

chart.type = "col" 

chart.style = 10 

chart.title = "Net Lift $ by Month (Actual vs. Expected at Pre-EMR Mix)" 

chart.y_axis.title = "Net Lift ($)" 

chart.x_axis.title = "Month" 

chart.height = 9 

chart.width = 24

 

data = Reference(smd, min_col=2, max_col=12, min_row=25, max_row=25) 

cats = Reference(smd, min_col=2, max_col=12, min_row=5, max_row=5) 

chart.add_data(data, titles_from_data=False) 

chart.set_categories(cats)

 

s = chart.series[0] 

s.graphicalProperties.solidFill = "4CAF50" 

chart.dataLabels = DataLabelList() 

chart.dataLabels.showVal = True 

chart.dataLabels.numFmt = '$#,##0' 

chart.legend = None

 

smd.add_chart(chart, "B29")

 

# --------------------------------------------------------------- 

# STEP 4: Summary - Dollars (high-level recap) 

# --------------------------------------------------------------- 

if "Summary - Dollars" in wb.sheetnames: 

    del wb["Summary - Dollars"] 

sd = wb.create_sheet("Summary - Dollars") 

sd["A1"] = "2024 - 2026 CC/MCC IP Visits — Financial Impact Summary" 

sd["A1"].font = Font(bold=True, size=12) 

sd["A3"] = "Summary - Dollars (Post-EMR: Nov 2025 - Sep 2026)" 

sd["A3"].font = Font(bold=True)

 

headers = ["Grouping","Blended $/Case","Pre-EMR Mix %","Expected Volume (11-Mo)", 

           "Expected $","Actual Mix % (Post-EMR Avg)","Actual Volume (11-Mo)", 

           "Actual $","Net Lift $"] 

for i, h in enumerate(headers): 

    c = sd.cell(row=5, column=i+1, value=h) 

    c.font = bold 

    c.fill = header_fill 

    c.alignment = Alignment(wrap_text=True)

 

rate_cells = {"MCC": "D4", "CC": "D5", "W/O": "D6"} 

summary_pre_rows = {"MCC": 6, "CC": 7, "W/O": 8} 

smd_rows = {"MCC": (11,20,6,15), "CC": (12,21,7,16), "W/O": (13,22,8,17)}

 

for i, tier in enumerate(["MCC","CC","W/O"]): 

    r = 6 + i 

    sr = summary_pre_rows[tier] 

    exp_vol_row, exp_d_row, act_vol_row, act_d_row = smd_rows[tier] 

    sd.cell(row=r, column=1, value=tier) 

    sd.cell(row=r, column=2, value=f"=Rates!{rate_cells[tier]}") 

    sd.cell(row=r, column=3, value=f"=Summary!B{sr}") 

    sd.cell(row=r, column=4, value=f"='Summary by Month - Dollars'!M{exp_vol_row}") 

    sd.cell(row=r, column=5, value=f"='Summary by Month - Dollars'!M{exp_d_row}") 

    sd.cell(row=r, column=6, value=f"=Summary!C{sr}") 

    sd.cell(row=r, column=7, value=f"='Summary by Month - Dollars'!M{act_vol_row}") 

    sd.cell(row=r, column=8, value=f"='Summary by Month - Dollars'!M{act_d_row}") 

    sd.cell(row=r, column=9, value=f"=H{r}-E{r}")

 

    sd.cell(row=r, column=2).number_format = dollar_fmt 

    sd.cell(row=r, column=3).number_format = pct_fmt 

    sd.cell(row=r, column=4).number_format = num_fmt 

    sd.cell(row=r, column=5).number_format = dollar_fmt 

    sd.cell(row=r, column=6).number_format = pct_fmt 

    sd.cell(row=r, column=7).number_format = num_fmt 

    sd.cell(row=r, column=8).number_format = dollar_fmt 

    sd.cell(row=r, column=9).number_format = dollar_fmt

 

# Total row 

sd.cell(row=9, column=1, value="Total").font = bold 

for col in [4,5,7,8,9]: 

    letter = get_column_letter(col) 

    sd.cell(row=9, column=col, value=f"=SUM({letter}6:{letter}8)") 

    sd.cell(row=9, column=col).font = bold 

    sd.cell(row=9, column=col).number_format = num_fmt if col in (4,7) else dollar_fmt

 

# Annualized run-rate 

sd["A11"] = "Annualized Lift Run-Rate (Total Lift ÷ 11 × 12)" 

sd["A11"].font = bold 

sd["I11"] = "=I9/11*12" 

sd["I11"].font = Font(bold=True, color="006100") 

sd["I11"].number_format = dollar_fmt

 

sd.column_dimensions["A"].width = 10 

for col, w in zip("BCDEFGHI", [16,14,20,16,22,18,16,16]): 

    sd.column_dimensions[col].width = w

 

# --------------------------------------------------------------- 

wb.save(OUT_FILE) 

print(f"Saved: {OUT_FILE}") 

Saved: L:/2026 Pilar Plant Files/Requests from Departments/2024-2026 CC-MCC (CJ) v3 - with Dollars.xlsx
